# 15 - Phase 6: uncertainty-aware / selective prediction

**Main question**: can uncertainty identify predictions that are likely to be wrong, particularly
under external domain shift? This connects three already-completed pieces of work - Phase 4's MC
Dropout + calibration, Phase 5's external IQ-OTH/NCCD evaluation, and the checkpoint-identification
conventions established in Phases 2-5 - into one final experiment.

**No retraining, no new models.** Every model here is loaded from an existing `.keras` checkpoint;
every forward pass (deterministic or MC Dropout's stochastic-but-still-forward-only passes) is
inference. There is no `.fit()` call anywhere in this notebook or in `src/uncertainty_utils.py`.

**Nothing in this notebook has been executed.** It was written, not run - the checkpoints, the
external dataset, and a full TensorFlow environment all live only on the machine that will actually
run this.

---

## Checkpoints this notebook depends on

| Role | Checkpoint | How it's identified |
|---|---|---|
| Primary | `checkpoints_local/miniconvnet_single_run.keras` | Fixed name, same instance as Phases 3-4-5 |
| Secondary | `checkpoints_local/vgg16_runB_finetuned.keras` (as of the last Phase 2 run) | **Read programmatically** from `results/fair_baseline/metrics_fair_baseline.csv` every time this runs - never hardcoded |

**The leakage-controlled MiniConvNet question, resolved in advance and disclosed prominently below
(Step 0b)**: the leakage-controlled (Option B) 3-fold CV never produced a standalone saved
checkpoint - confirmed by inspecting `src/leakage_cv_utils.py` and `notebooks/14_leakage_controlled_cv.ipynb`
directly (neither contains a single `.save()` call; Option B was built to report aggregate per-fold
metrics only). This notebook therefore uses the single-run checkpoint for every MiniConvNet result
and states that substitution explicitly wherever a MiniConvNet number appears - it is a disclosed,
deliberate choice, not a shortcut being hidden.

## Scope decisions carried over from earlier phases, unchanged

* **External label mapping**: reused as-is from Phase 5 -
  `EXTERNAL_TASK_FRAMINGS["malignant_vs_normal_excl_benign"]` (benign excluded from scoring). Not
  modified here.
* **MC Dropout mechanism**: reused as-is from Phase 4 - `calibration_utils.mc_dropout_raw_passes`
  (the stochastic-forward-pass core extracted from `mc_dropout_predict` specifically so this phase
  could reuse it without duplicating the loop). `mc_dropout_predict`'s own behaviour for existing
  callers (notebook 12) is unchanged by that refactor.
* **High-confidence threshold**: reused as-is from Phase 3 - `gradcam_utils.HIGH_CONFIDENCE_THRESHOLD`
  (0.75), not picked fresh for this phase.
* **n_passes = 15**: within the requested 10-20 range, and the exact value Phase 4 already used and
  measured the cost of (9.0s total for MiniConvNet, 264.8s for VGG16, both over 315 internal images) -
  reused rather than re-chosen.

**Why internal MC Dropout is re-run here rather than only read from `results/calibration/mc_dropout_results.json`**:
that file only persisted AGGREGATE statistics (mean entropy, mean confidence variance) - Phase 4's
scope never asked for per-image arrays. This phase's Step 4/6/7 analyses need per-image entropy, so
internal MC Dropout is re-run once, on the identical checkpoint/split/seed/pass-count, purely for that
granularity. This is inference-only re-computation, not retraining, and a consistency check against
the already-recorded aggregate numbers is run immediately after (Step 3), so any unexpected drift is
caught rather than assumed away.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import json
import time
import numpy as np
import pandas as pd

from src.config import *
from src.data_utils import load_split, make_dataset

from src.gradcam_utils import (identify_strongest_finetuned_baseline, verify_required_checkpoints,
                               HIGH_CONFIDENCE_THRESHOLD)
from src.finetune_utils import CHECKPOINTS_LOCAL

from src.calibration_utils import (
    ensure_calibration_dirs, locate_or_generate_predictions,
    has_batchnorm, mc_dropout_predict, mc_dropout_raw_passes,
    plot_confidence_distribution, plot_confidence_by_correctness,
    CALIBRATION_METRICS_CSV,
)

from src.external_eval_utils import (
    ensure_external_eval_dirs, resolve_external_data_root, index_external_dataset,
    external_class_counts, make_external_dataset, reduce_to_binary_tumor_probs,
    EXTERNAL_TASK_FRAMINGS, EXTERNAL_CLASS_NAMES, print_external_eval_time_estimate,
    load_in_domain_binary_reference, domain_shift_summary,
)

from src.uncertainty_utils import (
    ensure_uncertainty_dirs, UNCERTAINTY_DIR,
    check_leakage_controlled_checkpoint, identify_models_for_this_phase,
    deterministic_external_predictions, mc_dropout_external, internal_tumor_recall,
    correct_vs_incorrect_stats, uncertainty_bins, DEFAULT_UNCERTAINTY_BIN_LABELS,
    risk_coverage_curve, DEFAULT_COVERAGE_LEVELS,
    confidence_vs_entropy_table, high_confidence_incorrect,
    plot_entropy_by_correctness, plot_risk_coverage_curve,
    plot_internal_vs_external_entropy, plot_internal_vs_external_accuracy_recall,
    plot_high_confidence_incorrect_grid, gradcam_overlays_for_external_failures,
    build_final_results_table, save_final_results_table,
    save_uncertainty_predictions, record_uncertainty_result, load_uncertainty_results,
    save_selective_prediction_results, SELECTIVE_PREDICTION_CSV,
)

pd.set_option('display.width', 200)
print('uncertainty base dir:', UNCERTAINTY_DIR)

## STEP 1 (checkpoints) - identify models, verify checkpoints, disclose the substitution

In [ ]:
ids = identify_models_for_this_phase()
print('PRIMARY  :', ids['primary_model_name'], '->', ids['primary_checkpoint'])
if ids['secondary_available']:
    print('SECONDARY:', ids['secondary_model_name'], '->', ids['secondary_checkpoint'])
else:
    print('SECONDARY: could not be determined -', ids['secondary_reason_unavailable'])

PRIMARY_MODEL_NAME = ids['primary_model_name']
PRIMARY_RUN_NAME = ids['primary_run_name']
PRIMARY_CHECKPOINT = ids['primary_checkpoint']
SECONDARY_MODEL_NAME = ids['secondary_model_name']
SECONDARY_RUN_NAME = ids['secondary_run_name']
SECONDARY_CHECKPOINT = ids['secondary_checkpoint']

In [ ]:
ck_status = verify_required_checkpoints(
    PRIMARY_CHECKPOINT, SECONDARY_CHECKPOINT,
    secondary_label=f"{SECONDARY_MODEL_NAME or 'secondary'} (strongest Phase 2 baseline)")

print(ck_status['primary']['message'])
print(ck_status['secondary']['message'])

PRIMARY_OK = ck_status['primary']['exists']
SECONDARY_OK = ck_status['secondary']['exists']
print()
print('PRIMARY_OK  :', PRIMARY_OK)
print('SECONDARY_OK:', SECONDARY_OK)
if not PRIMARY_OK:
    print()
    print('STOP: MiniConvNet checkpoint is missing. This notebook does NOT retrain to produce one.')
    print('Restore it from the manual backup made after Phase 2/notebook 10 before continuing.')
if not SECONDARY_OK:
    print()
    print(f'NOTE: {SECONDARY_MODEL_NAME or "secondary model"} checkpoint missing or undetermined.')
    print('Secondary-model sections below will be skipped; MiniConvNet-only analysis can proceed.')

## STEP 0b - the leakage-controlled-checkpoint disclosure (read this before trusting any MiniConvNet number below)

In [ ]:
leakage_ck = check_leakage_controlled_checkpoint()
print('leakage-controlled checkpoint candidates found:', leakage_ck['candidates'] or 'NONE')
print('substitution required:', leakage_ck['substitution_required'])
print()
if leakage_ck['disclosure']:
    print(leakage_ck['disclosure'])

CHECKPOINT_SUBSTITUTION_NOTE = leakage_ck['disclosure'] or ''

## STEP 2 - the external dataset (existing pipeline, unchanged preprocessing and label mapping)

Reusing `resolve_external_data_root()`, `index_external_dataset()`, `make_external_dataset()` exactly
as Phase 5 built them - no preprocessing differences introduced here, per the brief's instruction.

In [ ]:
external_root = resolve_external_data_root()
print('external data root:', external_root)

external_df = index_external_dataset(external_root)
print('external images indexed:', len(external_df))
print(external_class_counts(external_df).to_string())

## Internal data (existing `faithful` split, unchanged)

In [ ]:
sdf = load_split('faithful')
internal_test_frame = sdf[sdf['split'] == 'test'].reset_index(drop=True)
print('internal faithful test images:', len(internal_test_frame))
print(internal_test_frame['class'].value_counts().reindex(CLASS_NAMES).to_string())

## STEP 3a - internal deterministic predictions (reused, not regenerated if a saved file already matches)

`locate_or_generate_predictions()` is Phase 4's own exact-match-or-forward-pass logic (never
substitutes a different run's saved numbers) - reused verbatim here for the "deterministic
prediction / deterministic confidence" columns Step 3 asks for.

In [ ]:
det_internal = {}
if PRIMARY_OK:
    det_internal[PRIMARY_MODEL_NAME] = locate_or_generate_predictions(
        PRIMARY_RUN_NAME, PRIMARY_CHECKPOINT, sdf, 'test', one_hot=True,
        model_short_name='miniconvnet')
if SECONDARY_OK:
    det_internal[SECONDARY_MODEL_NAME] = locate_or_generate_predictions(
        SECONDARY_RUN_NAME, SECONDARY_CHECKPOINT, sdf, 'test', one_hot=False,
        model_short_name=SECONDARY_MODEL_NAME.lower())

MODELS_AVAILABLE = list(det_internal.keys())
print('models with usable internal predictions:', MODELS_AVAILABLE)

## STEP 3b - internal MC Dropout, RE-RUN for per-image granularity (n_passes=15, consistency-checked)

Same checkpoint, same split, same seed, same pass count as Phase 4. Immediately after, this compares
the freshly computed mean entropy/accuracy against Phase 4's already-recorded aggregate
(`results/calibration/mc_dropout_results.json`) - if they diverge by more than a small tolerance,
that is reported as a discrepancy, not silently accepted.

In [ ]:
mc_internal = {}
for model_name in MODELS_AVAILABLE:
    checkpoint = PRIMARY_CHECKPOINT if model_name == PRIMARY_MODEL_NAME else SECONDARY_CHECKPOINT
    model = det_internal[model_name].get('model')
    if model is None:
        import tensorflow as tf
        print(f'{model_name}: loading checkpoint for MC Dropout (inference only): {checkpoint}')
        model = tf.keras.models.load_model(checkpoint)

    if has_batchnorm(model):
        print(f'{model_name}: SKIPPED - BatchNormalization present, MC Dropout not attempted.')
        continue

    one_hot = (model_name == PRIMARY_MODEL_NAME)
    ds = make_dataset(internal_test_frame, one_hot=one_hot)

    print(f'--- {model_name}: internal MC Dropout, 15 passes over {len(internal_test_frame)} images ---')
    t0 = time.time()
    _ = mc_dropout_predict(model, ds, n_passes=1, verbose=False)
    one_pass_seconds = time.time() - t0
    print(f'  first pass: {one_pass_seconds:.2f}s -> ~{one_pass_seconds*15:.1f}s projected for 15 passes')

    t0 = time.time()
    result = mc_dropout_predict(model, ds, n_passes=15, verbose=True)
    total_seconds = time.time() - t0
    print(f'  actual total: {total_seconds:.1f}s')
    result['total_seconds'] = total_seconds
    mc_internal[model_name] = result

    import tensorflow as tf
    tf.keras.backend.clear_session()

In [ ]:
# Consistency check against Phase 4's already-recorded aggregate.
TOLERANCE = 0.02
for model_name, result in mc_internal.items():
    fresh_mean_entropy = float(result['predictive_entropy'].mean())
    fresh_accuracy = float(result['correct'].mean())
    path = CALIBRATION_DIR = None
    from src.calibration_utils import MC_DROPOUT_RESULTS_JSON
    if MC_DROPOUT_RESULTS_JSON.exists():
        prior = json.loads(MC_DROPOUT_RESULTS_JSON.read_text()).get(model_name)
    else:
        prior = None
    print(f'{model_name}: fresh mean_entropy={fresh_mean_entropy:.4f}, fresh_accuracy={fresh_accuracy:.4f}')
    if prior:
        d_e = abs(fresh_mean_entropy - prior['mean_predictive_entropy'])
        d_a = abs(fresh_accuracy - prior['mean_accuracy'])
        print(f'  Phase 4 recorded: mean_entropy={prior["mean_predictive_entropy"]:.4f}, '
              f'accuracy={prior["mean_accuracy"]:.4f}  (deltas: {d_e:.4f}, {d_a:.4f})')
        if d_e > TOLERANCE or d_a > TOLERANCE:
            print(f'  !!! DIVERGES from Phase 4\'s recorded aggregate by more than {TOLERANCE} - '
                  'reported as a discrepancy, not assumed to be a bug and silently re-run.')
        else:
            print('  consistent with Phase 4\'s recorded aggregate (within tolerance).')
    else:
        print('  no prior aggregate found to compare against.')

## STEP 3c - external deterministic pass + external MC Dropout

The deterministic pass reuses `external_eval_utils.run_external_forward_pass()` unchanged (Phase 5's
own function - same code path that produced the already-reported 46.06%/56.09% external accuracies).
MC Dropout reuses the same stochastic core as the internal re-run, via `uncertainty_utils.mc_dropout_external()`,
which additionally reduces to the binary malignant-vs-normal task using Phase 5's own established
reduction and PRIMARY framing (benign excluded from scoring) - not a new label mapping.

In [ ]:
det_external = {}
mc_external = {}

for model_name in MODELS_AVAILABLE:
    checkpoint = PRIMARY_CHECKPOINT if model_name == PRIMARY_MODEL_NAME else SECONDARY_CHECKPOINT
    print(f'=== {model_name}: external deterministic pass ({len(external_df)} images) ===')
    t0 = time.time()
    external_label_det, y_prob_4class_det, model = deterministic_external_predictions(
        checkpoint, external_df)
    print(f'  wall clock: {time.time() - t0:.1f}s')
    det_external[model_name] = {'external_label': external_label_det, 'y_prob_4class': y_prob_4class_det}

    if has_batchnorm(model):
        print(f'  {model_name}: MC Dropout SKIPPED - BatchNormalization present.')
        continue

    ds = make_external_dataset(external_df)
    t0 = time.time()
    _ = mc_dropout_raw_passes(model, ds, n_passes=1, verbose=False)
    one_pass_seconds = time.time() - t0
    est_min = one_pass_seconds * 15 / 60
    print(f'  MC Dropout: first pass {one_pass_seconds:.2f}s -> ~{est_min:.1f} min projected for 15 passes')

    t0 = time.time()
    result = mc_dropout_external(model, external_df, n_passes=15, verbose=True)
    total_seconds = time.time() - t0
    print(f'  MC Dropout actual total: {total_seconds:.1f}s')
    result['total_seconds'] = total_seconds
    mc_external[model_name] = result

    import tensorflow as tf
    tf.keras.backend.clear_session()

## Assemble per-image tables and save (Step 13 reproducibility requirement)

For every (model, dataset) pair: deterministic prediction/confidence, MC mean probability, predicted
class, predictive entropy, confidence variance, correctness. Internal and external use different
label spaces, so their saved columns differ slightly - both are self-describing.

In [ ]:
internal_tables = {}
for model_name in MODELS_AVAILABLE:
    det = det_internal[model_name]
    mc = mc_internal.get(model_name)
    if mc is None:
        continue
    det_conf = det['y_prob'].max(axis=1)
    det_pred = det['y_pred']
    binary_mean_prob = reduce_to_binary_tumor_probs(mc['mean_probs'])
    eps = 1e-12
    entropy_binary = -np.sum(binary_mean_prob * np.log(binary_mean_prob + eps), axis=1)
    normal_idx = CLASS_NAMES.index(NORMAL_CLASS)

    frame = {
        'filepath': internal_test_frame['filepath'].values,
        'true_class': [CLASS_NAMES[i] for i in det['y_true']],
        'deterministic_pred_class': [CLASS_NAMES[i] for i in det_pred],
        'deterministic_confidence': det_conf,
        'mc_pred_class': [CLASS_NAMES[i] for i in mc['y_pred_mean']],
        'mc_predictive_entropy_4class': mc['predictive_entropy'],
        'mc_confidence_variance_4class': mc['confidence_variance'],
        'mc_prob_tumor_reduced': binary_mean_prob[:, 1],
        'mc_predictive_entropy_binary': entropy_binary,
        'true_binary_tumor': (det['y_true'] != normal_idx).astype(int),
        'mc_pred_binary_tumor': (mc['y_pred_mean'] != normal_idx).astype(int),
        'det_correct': (det_pred == det['y_true']),
        'mc_correct': mc['correct'],
    }
    # Full per-class MC mean probability, added so the saved per-image table is
    # complete on its own (report Table 7 / per_image_predictions.csv needs
    # this, not just the binary-reduced probability).
    for i, cls in enumerate(CLASS_NAMES):
        frame[f'mc_mean_prob_{cls}'] = mc['mean_probs'][:, i]
    save_uncertainty_predictions(model_name, 'internal', frame)
    internal_tables[model_name] = pd.DataFrame(frame)
    print(f'{model_name} (internal): {len(frame["filepath"])} rows saved')

In [ ]:
external_tables = {}
for model_name in MODELS_AVAILABLE:
    det = det_external[model_name]
    mc = mc_external.get(model_name)
    if mc is None:
        continue
    det_binary_prob = reduce_to_binary_tumor_probs(det['y_prob_4class'])
    det_pred_binary = det_binary_prob.argmax(axis=1)
    framing = EXTERNAL_TASK_FRAMINGS['malignant_vs_normal_excl_benign']
    scored_mask = framing['include'](det['external_label'])
    true_binary = framing['binary_true'](det['external_label'])

    frame = {
        'filepath': external_df['filepath'].values,
        'external_class': external_df['external_class'].values,
        'scored_mask': scored_mask,
        'true_binary_malignant': true_binary,
        'deterministic_confidence': det_binary_prob.max(axis=1),
        'deterministic_pred_binary_malignant': det_pred_binary,
        'mc_prob_malignant': mc['binary_mean_prob'][:, 1],
        'mc_pred_binary_malignant': mc['binary_pred'],
        'mc_predictive_entropy_binary': mc['predictive_entropy_binary'],
        'mc_predictive_entropy_4class': mc['predictive_entropy_4class'],
        'mc_confidence_variance_4class': mc['confidence_variance_4class'],
        'det_correct': (det_pred_binary == true_binary),
        'mc_correct': mc['correct'],
    }
    # Full per-class MC mean probability (our project's own 4-class softmax,
    # NOT the external 3-class label space) - added for the same completeness
    # reason as the internal frame above.
    for i, cls in enumerate(CLASS_NAMES):
        frame[f'mc_mean_prob_{cls}'] = mc['mean_probs_4class'][:, i]
    save_uncertainty_predictions(model_name, 'external', frame)
    external_tables[model_name] = pd.DataFrame(frame)
    print(f'{model_name} (external): {len(frame["filepath"])} rows saved '
          f'({int(scored_mask.sum())} scored, {int((~scored_mask).sum())} benign/excluded)')

## STEP 4 - uncertainty vs error

Mean/median entropy for correct vs incorrect, for every model, on BOTH datasets. Internal uses the
native 4-class predictive entropy (the actual internal decision space); external uses the
binary-reduced entropy (the actual external decision space, malignant vs normal) - each dataset's
own natural task, not a mismatched comparison. Cross-domain comparisons (Step 6) instead put both on
the SAME binary-reduced footing.

In [ ]:
cvi_internal, cvi_external = {}, {}
for model_name in MODELS_AVAILABLE:
    if model_name in internal_tables:
        t = internal_tables[model_name]
        cvi_internal[model_name] = correct_vs_incorrect_stats(t['mc_predictive_entropy_4class'], t['det_correct'])
        print(f'--- {model_name} (internal, native 4-class entropy) ---')
        print(json.dumps(cvi_internal[model_name], indent=2))
    if model_name in external_tables:
        t = external_tables[model_name]
        scored = t[t['scored_mask']]
        cvi_external[model_name] = correct_vs_incorrect_stats(
            scored['mc_predictive_entropy_binary'], scored['det_correct'])
        print(f'--- {model_name} (external, scored subset only, binary entropy) ---')
        print(json.dumps(cvi_external[model_name], indent=2))

In [ ]:
# FIGURE 1: entropy distribution, correct vs incorrect (internal + external, per model)
for model_name in MODELS_AVAILABLE:
    if model_name in internal_tables:
        t = internal_tables[model_name]
        plot_entropy_by_correctness(t['mc_predictive_entropy_4class'], t['det_correct'],
                                    model_name, 'internal', show=False)
    if model_name in external_tables:
        t = external_tables[model_name][external_tables[model_name]['scored_mask']]
        plot_entropy_by_correctness(t['mc_predictive_entropy_binary'], t['det_correct'],
                                    model_name, 'external', show=False)
print('Figure 1 (entropy by correctness) saved per model/dataset under results/uncertainty/<model>/figures/')

In [ ]:
# Confidence distribution (reused from Phase 4's own plotting function, unchanged).
for model_name in MODELS_AVAILABLE:
    t = internal_tables.get(model_name)
    if t is None:
        continue
    y_true = det_internal[model_name]['y_true']
    y_prob = det_internal[model_name]['y_prob']
    plot_confidence_distribution(y_true, y_prob, model_name, method_label='internal (Phase 6 reuse)', show=False)
print('confidence distributions saved to results/calibration/figures/ (reusing the Phase 4 function directly)')

In [ ]:
# Uncertainty bins (tercile quantiles of entropy, independent of correctness - see the module
# docstring for why this specific binning was chosen).
for model_name in MODELS_AVAILABLE:
    if model_name in internal_tables:
        t = internal_tables[model_name]
        print(f'--- {model_name} (internal) uncertainty bins ---')
        print(uncertainty_bins(t['mc_predictive_entropy_4class'], t['det_correct']).to_string(index=False))
    if model_name in external_tables:
        t = external_tables[model_name][external_tables[model_name]['scored_mask']]
        print(f'--- {model_name} (external, scored) uncertainty bins ---')
        print(uncertainty_bins(t['mc_predictive_entropy_binary'], t['det_correct']).to_string(index=False))

## STEP 5 - selective prediction / risk-coverage

Coverage levels are the standard 100%-to-10% decile grid named in the brief - not searched over to
find a flattering point. Least-uncertain predictions are retained first at each level; malignant
recall (external) / tumour recall (internal) among the RETAINED predictions is reported alongside
accuracy, since accuracy alone can hide a recall collapse.

In [ ]:
rc_tables = {}
normal_idx = CLASS_NAMES.index(NORMAL_CLASS)

for model_name in MODELS_AVAILABLE:
    if model_name in internal_tables:
        t = internal_tables[model_name]
        rc_tables[(model_name, 'internal')] = risk_coverage_curve(
            t['mc_predictive_entropy_4class'], t['det_correct'],
            y_true_binary=t['true_binary_tumor'], y_pred_binary=t['mc_pred_binary_tumor'],
            positive_label=1)
    if model_name in external_tables:
        t = external_tables[model_name][external_tables[model_name]['scored_mask']].reset_index(drop=True)
        rc_tables[(model_name, 'external')] = risk_coverage_curve(
            t['mc_predictive_entropy_binary'], t['det_correct'],
            y_true_binary=t['true_binary_malignant'], y_pred_binary=t['mc_pred_binary_malignant'],
            positive_label=1)

for (model_name, dataset_label), rc in rc_tables.items():
    print(f'--- {model_name} ({dataset_label}) risk-coverage ---')
    print(rc.to_string(index=False))
    print()

In [ ]:
# FIGURE 2 (MiniConvNet) / FIGURE 3 (VGG16): risk-coverage curves, external (the primary
# domain-shift-relevant curve - the brief's Figures 2/3 are specified per model, using the
# external evaluation since that is the setting the brief's Step 5/6 are actually about).
for model_name in MODELS_AVAILABLE:
    if (model_name, 'external') in rc_tables:
        plot_risk_coverage_curve(rc_tables[(model_name, 'external')], model_name, 'external', show=False)
    if (model_name, 'internal') in rc_tables:
        plot_risk_coverage_curve(rc_tables[(model_name, 'internal')], model_name, 'internal', show=False)
print('risk-coverage figures saved under results/uncertainty/<model>/figures/')

save_selective_prediction_results(rc_tables)
print('selective-prediction table saved to', SELECTIVE_PREDICTION_CSV)

## STEP 6 - external domain-shift analysis

Internal vs external accuracy, AUC, malignant/tumour recall, confidence, and entropy - all on the
SAME binary-reduced footing (tumour-vs-normal internally, malignant-vs-normal externally, both via
the identical reduction), which is the one task genuinely defined in both domains.

In [ ]:
domain_rows = []
for model_name in MODELS_AVAILABLE:
    it = internal_tables.get(model_name)
    et = external_tables.get(model_name)
    if it is None or et is None:
        continue
    et_scored = et[et['scored_mask']]

    internal_accuracy = float(it['det_correct'].mean())
    external_accuracy = float(et_scored['det_correct'].mean())
    internal_recall = internal_tumor_recall(det_internal[model_name]['y_true'], det_internal[model_name]['y_pred'])

    from src.external_eval_utils import EXTERNAL_METRICS_CSV
    ext_metrics = pd.read_csv(EXTERNAL_METRICS_CSV) if EXTERNAL_METRICS_CSV.exists() else pd.DataFrame()
    row = ext_metrics[(ext_metrics['model'] == model_name)
                      & (ext_metrics['framing'] == 'malignant_vs_normal_excl_benign')]
    external_recall = float(row.iloc[0]['recall_tumor']) if len(row) else float('nan')
    external_auc = float(row.iloc[0]['auc']) if len(row) and pd.notna(row.iloc[0]['auc']) else None

    internal_mean_conf = float(det_internal[model_name]['y_prob'].max(axis=1).mean())
    external_mean_conf = float(et_scored['deterministic_confidence'].mean())
    internal_mean_entropy_binary = float(it['mc_predictive_entropy_binary'].mean())
    external_mean_entropy_binary = float(et_scored['mc_predictive_entropy_binary'].mean())

    print(f'=== {model_name}: internal vs external ===')
    print(f'  accuracy       : {internal_accuracy:.4f} -> {external_accuracy:.4f}  '
          f'(drop {internal_accuracy - external_accuracy:+.4f})')
    print(f'  AUC (external) : {external_auc}')
    print(f'  recall (tumour/malignant): {internal_recall:.4f} -> {external_recall:.4f}')
    print(f'  mean confidence: {internal_mean_conf:.4f} -> {external_mean_conf:.4f}')
    print(f'  mean entropy (binary-reduced): {internal_mean_entropy_binary:.4f} -> {external_mean_entropy_binary:.4f}  '
          f'({"HIGHER" if external_mean_entropy_binary > internal_mean_entropy_binary else "not higher"} externally)')
    print()

    domain_rows.append({'model': model_name, 'internal_accuracy': internal_accuracy,
                        'external_accuracy': external_accuracy, 'external_auc': external_auc,
                        'internal_recall': internal_recall, 'external_recall': external_recall,
                        'internal_mean_confidence': internal_mean_conf, 'external_mean_confidence': external_mean_conf,
                        'internal_mean_entropy_binary': internal_mean_entropy_binary,
                        'external_mean_entropy_binary': external_mean_entropy_binary})

domain_shift_df = pd.DataFrame(domain_rows)
print(domain_shift_df.round(4).to_string(index=False))

In [ ]:
# FIGURE 5 + FIGURE 6
for model_name in MODELS_AVAILABLE:
    it = internal_tables.get(model_name)
    et = external_tables.get(model_name)
    if it is None or et is None:
        continue
    et_scored = et[et['scored_mask']]
    plot_internal_vs_external_entropy(it['mc_predictive_entropy_binary'], et_scored['mc_predictive_entropy_binary'],
                                      model_name, show=False)

    row = domain_shift_df[domain_shift_df['model'] == model_name].iloc[0]
    plot_internal_vs_external_accuracy_recall(
        model_name, row['internal_accuracy'], row['external_accuracy'],
        row['internal_recall'], row['external_recall'], show=False)
print('Figures 5 and 6 saved under results/uncertainty/<model>/figures/')

**Caution on interpretation (per the brief's Step 6 instruction)**: an increase in mean entropy on
external data, if observed above, is reported as *"uncertainty showed a useful association with the
domain shift"* at most - never as *"uncertainty detects domain shift"* outright. Whether that
association is strong enough to be useful in practice depends on the risk-coverage numbers in Step 5,
not on this comparison alone.

## STEP 7 - confidence vs MC uncertainty; high-confidence incorrect predictions

Deterministic softmax confidence and MC predictive entropy are not necessarily the same signal - this
section checks how closely they actually agree, and specifically isolates the "dangerous" case: high
deterministic confidence that turned out to be wrong.

In [ ]:
hc_tables = {}
for model_name in MODELS_AVAILABLE:
    if model_name in internal_tables:
        t = internal_tables[model_name]
        cve = confidence_vs_entropy_table(t['deterministic_confidence'], t['mc_predictive_entropy_4class'], t['det_correct'])
        corr = cve[['deterministic_confidence', 'mc_predictive_entropy']].corr().iloc[0, 1]
        print(f'{model_name} (internal): confidence vs entropy correlation = {corr:.4f} '
              '(expect negative - higher confidence, lower entropy, if the two agree)')
        hc = high_confidence_incorrect(t['deterministic_confidence'], t['mc_predictive_entropy_4class'],
                                       t['det_correct'], filepath=t['filepath'],
                                       true_label=t['true_class'], pred_label=t['deterministic_pred_class'])
        hc_tables[(model_name, 'internal')] = hc
        print(f'  high-confidence (>= {HIGH_CONFIDENCE_THRESHOLD}) INCORRECT: {len(hc)} of {len(t)}')
        if len(hc):
            print(f'    their MC entropy: mean={hc["mc_predictive_entropy"].mean():.4f}, '
                  f'median={hc["mc_predictive_entropy"].median():.4f}')

    if model_name in external_tables:
        t = external_tables[model_name][external_tables[model_name]['scored_mask']].reset_index(drop=True)
        cve = confidence_vs_entropy_table(t['deterministic_confidence'], t['mc_predictive_entropy_binary'], t['det_correct'])
        corr = cve[['deterministic_confidence', 'mc_predictive_entropy']].corr().iloc[0, 1]
        print(f'{model_name} (external, scored): confidence vs entropy correlation = {corr:.4f}')
        hc_ext = high_confidence_incorrect(t['deterministic_confidence'], t['mc_predictive_entropy_binary'],
                                           t['det_correct'], filepath=t['filepath'],
                                           true_label=t['true_binary_malignant'],
                                           pred_label=t['deterministic_pred_binary_malignant'])
        hc_tables[(model_name, 'external')] = hc_ext
        print(f'  high-confidence (>= {HIGH_CONFIDENCE_THRESHOLD}) INCORRECT: {len(hc_ext)} of {len(t)} '
              '(these are potentially dangerous overconfident external failures)')
        if len(hc_ext):
            print(f'    their MC entropy: mean={hc_ext["mc_predictive_entropy"].mean():.4f}, '
                  f'median={hc_ext["mc_predictive_entropy"].median():.4f}')

In [ ]:
# FIGURE 4: confidence vs correctness (reused directly from Phase 4's own function, unchanged).
for model_name in MODELS_AVAILABLE:
    y_true = det_internal[model_name]['y_true']
    y_prob = det_internal[model_name]['y_prob']
    plot_confidence_by_correctness(y_true, y_prob, model_name, method_label='internal', show=False)

    if model_name in external_tables:
        # Reconstruct [p_normal, p_tumor] directly from the saved 4-class probabilities via the
        # same reduction used everywhere else in this notebook - not re-derived from confidence.
        et_full = external_tables[model_name]
        det_binary_prob_full = reduce_to_binary_tumor_probs(det_external[model_name]['y_prob_4class'])
        scored_mask = et_full['scored_mask'].to_numpy()
        plot_confidence_by_correctness(et_full['true_binary_malignant'][scored_mask].to_numpy(),
                                       det_binary_prob_full[scored_mask], model_name,
                                       method_label='external (malignant-vs-normal)', show=False)
print('Figure 4 (confidence vs correctness) saved to results/calibration/figures/ for both datasets')

## Optional: fresh Grad-CAM on the external high-confidence-incorrect images

No Grad-CAM output from Phase 3 covers the external dataset (it only ever analysed internal CT test
images), so there is nothing pre-existing to connect to. This generates a SMALL number of fresh
overlays (reusing Phase 3's nested-backbone-aware Grad-CAM machinery, already fixed for VGG16) purely
to visually inspect the most concerning external failures - optional, best-effort, and skipped
cleanly (with a printed reason) if it fails for a given model rather than forcing unreliable output.

In [ ]:
RUN_OPTIONAL_GRADCAM_ON_EXTERNAL_FAILURES = True
gradcam_overlays = {}

if RUN_OPTIONAL_GRADCAM_ON_EXTERNAL_FAILURES:
    for model_name in MODELS_AVAILABLE:
        hc = hc_tables.get((model_name, 'external'))
        if hc is None or hc.empty:
            print(f'{model_name}: no external high-confidence-incorrect cases - nothing to visualise.')
            continue
        checkpoint = PRIMARY_CHECKPOINT if model_name == PRIMARY_MODEL_NAME else SECONDARY_CHECKPOINT
        import tensorflow as tf
        model = tf.keras.models.load_model(checkpoint)
        overlays = gradcam_overlays_for_external_failures(model, hc, max_n=8)
        gradcam_overlays[model_name] = overlays
        print(f'{model_name}: {len(overlays)} Grad-CAM overlay(s) generated for external failures')
        tf.keras.backend.clear_session()
else:
    print('Optional Grad-CAM-on-external-failures section disabled.')

## STEP 8 - Figure 7: high-confidence incorrect external predictions

In [ ]:
for model_name in MODELS_AVAILABLE:
    hc = hc_tables.get((model_name, 'external'))
    if hc is None:
        continue
    plot_high_confidence_incorrect_grid(hc, model_name, max_display=8,
                                        gradcam_overlays=gradcam_overlays.get(model_name), show=False)
print('Figure 7 saved under results/uncertainty/<model>/figures/')

## Record per-model uncertainty summary rows (Step 13 reproducibility requirement)

In [ ]:
for model_name in MODELS_AVAILABLE:
    for dataset_label, table, cvi, entropy_col in (
        ('internal', internal_tables.get(model_name), cvi_internal.get(model_name), 'mc_predictive_entropy_4class'),
        ('external', (external_tables.get(model_name)[external_tables[model_name]['scored_mask']]
                      if model_name in external_tables else None),
         cvi_external.get(model_name), 'mc_predictive_entropy_binary'),
    ):
        if table is None or cvi is None:
            continue
        checkpoint = PRIMARY_CHECKPOINT if model_name == PRIMARY_MODEL_NAME else SECONDARY_CHECKPOINT
        hc = hc_tables.get((model_name, dataset_label))
        record_uncertainty_result({
            'model': model_name, 'dataset': dataset_label, 'n_samples': int(len(table)),
            'n_passes': 15, 'seed': SEED, 'checkpoint_used': checkpoint,
            'checkpoint_substitution_note': (CHECKPOINT_SUBSTITUTION_NOTE
                                             if model_name == PRIMARY_MODEL_NAME else ''),
            'mean_predictive_entropy': float(table[entropy_col].mean()),
            'median_predictive_entropy': float(table[entropy_col].median()),
            'mean_confidence_variance': float(table.get('mc_confidence_variance_4class', pd.Series(dtype=float)).mean())
                                        if 'mc_confidence_variance_4class' in table else None,
            'mean_entropy_correct': cvi['correct'].get('mean_entropy'),
            'mean_entropy_incorrect': cvi['incorrect'].get('mean_entropy'),
            'entropy_higher_for_incorrect': cvi.get('higher_for_incorrect'),
            'n_high_confidence_incorrect': int(len(hc)) if hc is not None else None,
            'high_confidence_threshold': HIGH_CONFIDENCE_THRESHOLD,
            'notes': f'n_passes=15, seed={SEED}, dataset={dataset_label}',
        })

print(load_uncertainty_results().to_string(index=False))

## STEP 9 - final consolidated results table

Every column traced live to its source file, or explicitly marked 'not available' where a metric
genuinely was not computed for that model (leakage-controlled CV is MiniConvNet-only).

In [ ]:
from src.models import build_miniconvnet, build_baseline
params_by_model = {}
if PRIMARY_MODEL_NAME in MODELS_AVAILABLE:
    params_by_model[PRIMARY_MODEL_NAME] = build_miniconvnet().count_params()
if SECONDARY_MODEL_NAME and SECONDARY_MODEL_NAME in MODELS_AVAILABLE:
    params_by_model[SECONDARY_MODEL_NAME] = build_baseline(SECONDARY_MODEL_NAME).count_params()

internal_uncertainty_stats = {}
for model_name in MODELS_AVAILABLE:
    if model_name in internal_tables:
        internal_uncertainty_stats[model_name] = {
            'mean_predictive_entropy': float(internal_tables[model_name]['mc_predictive_entropy_4class'].mean()),
            'correct_vs_incorrect': cvi_internal.get(model_name),
        }

risk_coverage_by_model = {m: rc_tables.get((m, 'external')) for m in MODELS_AVAILABLE}

final_table = build_final_results_table(MODELS_AVAILABLE, params_by_model,
                                        internal_uncertainty_stats, risk_coverage_by_model)
print(final_table.to_string(index=False))
save_final_results_table(final_table)

## STEP 10 - research interpretation

Answer each question directly from the numbers computed above - do not restate them more strongly
than the data supports, and do not hide a negative result.

**1. Does uncertainty distinguish correct and incorrect predictions?**
*(refer to Step 4's correct-vs-incorrect entropy stats and the confidence-vs-entropy correlation in
Step 7, for both datasets, both models)*

**2. Does uncertainty increase on external data?**
*(refer to Step 6's internal-vs-external mean entropy comparison; use "showed a useful association
with the domain shift" language, never "detects domain shift" unless the risk-coverage numbers
actually support that stronger claim)*

**3. Can selective prediction reduce error?**
*(refer to Step 5's risk-coverage tables/figures - does error rate fall as coverage decreases?)*

**4. Does selective prediction improve malignant recall?**
*(refer to the `positive_class_recall` column of the external risk-coverage table - does it rise,
fall, or stay flat as coverage decreases? A model that becomes more confident specifically about
'normal' calls under selection would show recall FALLING, not rising - report whichever actually
happened)*

**5. Which model is more reliable?**
*(weigh calibration (ECE/Brier, Step 9's table), external accuracy/AUC, and the uncertainty-vs-error
relationship together - a single number is not enough to answer this alone)*

**6. Are high-confidence wrong predictions still present?**
*(state the exact counts from Step 7, both datasets, both models - do not round down to "rare" if the
count is not actually small)*

**7. What does this imply about using lightweight CNNs under domain shift?**
*(a cautious, scoped conclusion - see Step 11 below for what NOT to claim)*

In [ ]:
print('=== DATA-DRIVEN ANSWERS TO STEP 10 (fill in the prose above from these) ===')
for model_name in MODELS_AVAILABLE:
    print(f'--- {model_name} ---')
    if model_name in cvi_internal:
        print('  Q1 (internal):', cvi_internal[model_name]['higher_for_incorrect'],
              '- delta =', cvi_internal[model_name]['mean_entropy_delta_incorrect_minus_correct'])
    if model_name in cvi_external:
        print('  Q1 (external):', cvi_external[model_name]['higher_for_incorrect'],
              '- delta =', cvi_external[model_name]['mean_entropy_delta_incorrect_minus_correct'])
    row = domain_shift_df[domain_shift_df['model'] == model_name]
    if len(row):
        r = row.iloc[0]
        print(f'  Q2: internal entropy {r["internal_mean_entropy_binary"]:.4f} -> '
              f'external {r["external_mean_entropy_binary"]:.4f}')
    rc = rc_tables.get((model_name, 'external'))
    if rc is not None:
        print('  Q3/Q4 (external risk-coverage):')
        print('   ', rc[['coverage', 'accuracy', 'error_rate', 'positive_class_recall']].to_string(index=False))
    hc_i = hc_tables.get((model_name, 'internal'))
    hc_e = hc_tables.get((model_name, 'external'))
    print(f'  Q6: high-confidence-incorrect - internal={len(hc_i) if hc_i is not None else "n/a"}, '
          f'external={len(hc_e) if hc_e is not None else "n/a"}')

## STEP 11 - do not overclaim

This is a research evaluation, not a clinical diagnostic system. **This notebook does not claim, and
its results must never be represented as showing**:

* clinical readiness or clinical safety of any model here,
* patient-level generalisation - no patient identifiers were ever available for the primary CT
  dataset (confirmed in the Phase 1 leakage analysis), so nothing here can speak to how these models
  would behave across genuinely distinct patients,
* real-world diagnostic validity - the external dataset uses a different label scheme (benign /
  malignant / normal vs this project's 4-class NSCLC subtype scheme), reduced to a binary task for
  comparison; that reduction is a research convenience, not a validated clinical equivalence.

Findings below should be read as: **experimental findings** (the numbers actually computed above),
**possible explanations** (reasoned but not proven interpretations of those numbers), **limitations**
(Step 12/13), and **future work** - kept visibly separate, not blended into one confident narrative.

## STEP 12/13 - CPU constraint and reproducibility, as executed in this notebook

* MC Dropout: 15 passes (within the requested 10-20 range), reused from Phase 4, not re-chosen.
* No hyperparameter search anywhere in this notebook.
* Every checkpoint is loaded once per model per dataset; no retraining.
* All per-image predictions, uncertainty values, configs, and figures are saved (see the file list in
  the final cell) so this analysis need not be repeated to reproduce its numbers.
* Existing results are read, never overwritten: `outputs/`, `results/fair_baseline/`,
  `results/calibration/`, `results/external_eval/`, `results/gradcam/` are all read-only inputs to
  this notebook.

In [ ]:
# Final file/path checklist and summary block.
print('PHASE:                Phase 6 - uncertainty-aware / selective prediction')
print('MODELS:               ' + ', '.join(MODELS_AVAILABLE))
print('DATASETS:              internal (faithful test split), external (IQ-OTH/NCCD)')
print('CHECKPOINT SUBSTITUTION:')
print('   ', CHECKPOINT_SUBSTITUTION_NOTE if CHECKPOINT_SUBSTITUTION_NOTE else 'none needed')
print('N_MC_PASSES:           15')
print('KEY FINDINGS:          fill in from Step 10\'s data-driven answers above')
print('FILES CREATED:')
print(f'   {UNCERTAINTY_DIR}/uncertainty_metrics.csv / .json')
print(f'   {UNCERTAINTY_DIR}/selective_prediction_results.csv')
print(f'   {UNCERTAINTY_DIR}/final_results_table.csv / .json')
for model_name in MODELS_AVAILABLE:
    print(f'   {UNCERTAINTY_DIR}/{model_name.lower()}/predictions_used/ (2 files: internal, external)')
    print(f'   {UNCERTAINTY_DIR}/{model_name.lower()}/figures/ (Figures 1-7 subset applicable to this model)')
print('CPU TIME:              sum of the per-step wall-clock prints above '
      '(dominant cost: VGG16 external MC Dropout, ~15 min projected from Phase 4\'s measured rate)')
print('LIMITATIONS:           see Step 11 above; additionally: leakage-controlled MiniConvNet results')
print('                       use a substitute checkpoint (disclosed above); external label scheme')
print('                       differs from the internal one and is reduced to a binary task for')
print('                       comparison; sample sizes are fixed by the existing datasets, not chosen.')